In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

In [ ]:
# crea las tablas

In [5]:
query = '''
CREATE TABLE IF NOT EXISTS CERVEZAS (
    CodC TEXT PRIMARY KEY,
    Envase TEXT,
    Capacidad REAL,
    Stock INTEGER
);
'''

crsr.execute(query)

In [6]:
query = '''
CREATE TABLE IF NOT EXISTS BARES (
    CodB TEXT PRIMARY KEY,
    Nombre TEXT NOT NULL,
    Cif TEXT,
    Localidad TEXT
);
'''

crsr.execute(query)

In [7]:
query = '''
CREATE TABLE IF NOT EXISTS EMPLEADOS (
    CodE INTEGER PRIMARY KEY,
    Nombre TEXT NOT NULL,
    Sueldo INTEGER
);
'''

crsr.execute(query)

In [8]:
query = '''
CREATE TABLE REPARTO (
    CodE INTEGER,
    CodB TEXT,
    CodC TEXT,
    Fecha DATE,
    Cantidad INTEGER,
    FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
    FOREIGN KEY (CodB) REFERENCES BARES(CodB),
    FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
'''
crsr.execute(query)

In [9]:
query = '''
INSERT OR REPLACE INTO CERVEZAS (CodC, Envase, Capacidad, Stock)
VALUES
('01', 'Botella', 0.20, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1.00, 288),
('05', 'Barril', 60.00, 30);
'''
crsr.execute(query)

In [10]:
query = '''
INSERT OR REPLACE INTO BARES (CodB, Nombre, Cif, Localidad)
VALUES
('001', 'Stop', '11111111X', 'Villa Botijo'),
('002', 'Las Vegas', '22222222Y', 'Villa Botijo'),
('003', 'Club Social', NULL, 'Las Ranas'),
('004', 'Otra Ronda', '33333333Z', 'La Esporja');
'''
# CRUD Create Register(insert) Update Delete
crsr.execute(query)

In [11]:
query = '''
INSERT OR REPLACE INTO EMPLEADOS (CodE, Nombre, Sueldo)
VALUES
(1, 'Carlos Lopez', 120000),
(2, 'Rosa Perez', 110000),
(3, 'Luisa Garcia', 100000);
'''
crsr.execute(query)

In [12]:
query = '''
INSERT OR REPLACE INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
'''
crsr.execute(query)

In [13]:
query = '''
SELECT * FROM empleados;
'''
crsr.execute(query)

In [14]:
query = '''
SELECT * FROM reparto;
'''
sql_query(query)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005-10-21,240
1,1,001,02,2005-10-21,48
2,1,002,03,2005-10-22,60
3,1,004,05,2005-10-22,4
4,2,002,03,2005-10-22,48
5,2,002,05,2005-10-23,2
6,2,004,01,2005-10-23,480
7,2,004,02,2005-10-24,72
8,3,003,03,2005-10-24,48
9,3,003,04,2005-10-25,20


In [15]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''
SELECT DISTINCT e.Nombre as "Nombre empleado", b.Nombre as "Nombre bar", r.Fecha
FROM  empleados e INNER JOIN reparto r
ON e.CodE = r.CodE
INNER JOIN bares b
ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop' AND r.Fecha  BETWEEN '2005-10-17' AND '2005-10-23'
'''

sql_query(query)

,Nombre empleado,Nombre bar,Fecha
0,Carlos Lopez,Stop,2005-10-21


In [16]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
SELECT DISTINCT BARES.Cif, BARES.Nombre
FROM BARES
JOIN REPARTO ON BARES.CodB = REPARTO.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
  AND CERVEZAS.Capacidad < 1
ORDER BY BARES.Localidad;
'''

sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [17]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad, REPARTO.Fecha, REPARTO.Cantidad
FROM REPARTO
JOIN EMPLEADOS ON REPARTO.CodE = EMPLEADOS.CodE
JOIN BARES ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
WHERE EMPLEADOS.Nombre = 'Carlos Lopez';
'''

sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


In [18]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''
SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad
FROM BARES
JOIN REPARTO ON BARES.CodB = REPARTO.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
  AND CERVEZAS.Capacidad IN (0.2, 0.33);
'''
sql_query(query)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


In [ ]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''
SELECT BARES.Nombre AS "Nombre bar",
       EMPLEADOS.Nombre AS "Nombre empleado",
       CERVEZAS.Envase
FROM EMPLEADOS
JOIN REPARTO ON EMPLEADOS.CodE = REPARTO.CodE
JOIN BARES ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
WHERE BARES.Nombre IN ('Stop', 'Las Vegas')
  AND CERVEZAS.Envase = 'Botella';
'''
sql_query(query)

,Nombre bar,Nombre empleado,Envase
0,Stop,Carlos Lopez,Botella
1,Stop,Carlos Lopez,Botella


In [20]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''
SELECT EMPLEADOS.CodE, EMPLEADOS.Nombre, COUNT(*) AS Viajes
FROM EMPLEADOS
JOIN REPARTO ON EMPLEADOS.CodE = REPARTO.CodE
JOIN BARES ON REPARTO.CodB = BARES.CodB
WHERE BARES.Localidad <> 'Villa Botijo'
GROUP BY EMPLEADOS.CodE, EMPLEADOS.Nombre;
'''
sql_query(query)

,CodE,Nombre,Viajes
0,1,Carlos Lopez,1
1,2,Rosa Perez,2
2,3,Luisa Garcia,2


In [ ]:
query = '''

'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esponja,359.76


In [21]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT BARES.Nombre, BARES.Localidad,
       SUM(REPARTO.Cantidad * CERVEZAS.Capacidad) AS litros
FROM BARES
JOIN REPARTO ON BARES.CodB = REPARTO.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
GROUP BY BARES.CodB, BARES.Nombre, BARES.Localidad
ORDER BY litros DESC
LIMIT 1;
'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esporja,359.76


In [22]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''
SELECT BARES.Nombre
FROM BARES
JOIN REPARTO ON BARES.CodB = REPARTO.CodB
JOIN CERVEZAS ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
  AND CERVEZAS.Capacidad < 1
GROUP BY BARES.CodB, BARES.Nombre
HAVING COUNT(DISTINCT CERVEZAS.CodC) = 2;
'''
sql_query(query)

,Nombre
0,Stop
1,Otra Ronda


In [23]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''
SELECT COUNT(DISTINCT REPARTO.Fecha) AS Dias_trabajados,
       EMPLEADOS.CodE,
       EMPLEADOS.Nombre
FROM REPARTO
JOIN EMPLEADOS ON REPARTO.CodE = EMPLEADOS.CodE
GROUP BY EMPLEADOS.CodE, EMPLEADOS.Nombre
ORDER BY Dias_trabajados DESC;
'''
sql_query(query)

,Dias_trabajados,CodE,Nombre
0,3,2,Rosa Perez
1,2,1,Carlos Lopez
2,2,3,Luisa Garcia


In [24]:
query = '''
SELECT EMPLEADOS.Sueldo * 1.05
FROM EMPLEADOS
WHERE EMPLEADOS.CodE = (
    SELECT REPARTO.CodE
    FROM REPARTO
    GROUP BY REPARTO.CodE
    ORDER BY COUNT(DISTINCT REPARTO.Fecha) DESC
    LIMIT 1
);
'''
sql_query(query)

,EMPLEADOS.Sueldo * 1.05
0,115500.0


In [25]:
query = '''
   SELECT REPARTO.CodE
FROM REPARTO
GROUP BY REPARTO.CodE
ORDER BY COUNT(DISTINCT REPARTO.Fecha) DESC
LIMIT 1;
'''
sql_query(query)

,CodE
0,2


In [26]:
query = '''
UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT REPARTO.CodE
    FROM REPARTO
    GROUP BY REPARTO.CodE
    ORDER BY COUNT(DISTINCT REPARTO.Fecha) DESC
    LIMIT 1
);
'''
crsr.execute(query)

In [27]:
query = '''
SELECT CodE, Nombre, Sueldo
FROM EMPLEADOS
WHERE CodE = 2;
'''
sql_query(query)

,CodE,Nombre,Sueldo
0,2,Rosa Perez,115500
